# Práctica Tema 4 – Gestión de Bases de Datos Avanzada
**Asignatura:** Programación Avanzada (GAB-2402)  
**Tecnológico de Estudios Superiores de Chalco**  
**Material creado por el Prof. Fabián Soberanes Martín**

---

## Objetivos
- Diseñar y crear un esquema relacional normalizado
- Cargar datos e imponer integridad referencial
- Escribir consultas avanzadas (JOIN, CTE, ventanas)
- Usar transacciones y CRUD seguro

> Completa las celdas `# TODO`. Usa SQLite (incluido en Python).

## 0. Configuración

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Ajusta si ejecutas desde otra carpeta
BASE = Path('..') if Path('../data').exists() else Path('.')
DATA = BASE / 'data'
SQL = BASE / 'sql'
DB_PATH = BASE / 'tienda.db'

print('DATA:', DATA.resolve())
print('Archivos CSV:', list(DATA.glob('*.csv')) if DATA.exists() else 'No encontrado')

---
## 1. Crear el esquema

In [ ]:
# Crear / reiniciar la base
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON')

schema_file = SQL / 'schema_tienda.sql'
if schema_file.exists():
    conn.executescript(schema_file.read_text(encoding='utf-8'))
    print('✅ Esquema cargado desde schema_tienda.sql')
else:
    # Fallback: crear tablas aquí
    conn.executescript('''
    CREATE TABLE categorias (
        categoria_id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL UNIQUE
    );
    CREATE TABLE productos (
        producto_id INTEGER PRIMARY KEY,
        categoria_id INTEGER REFERENCES categorias(categoria_id),
        nombre TEXT NOT NULL,
        precio REAL NOT NULL CHECK (precio >= 0),
        stock INTEGER NOT NULL DEFAULT 0 CHECK (stock >= 0)
    );
    CREATE TABLE clientes (
        cliente_id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email TEXT UNIQUE,
        ciudad TEXT
    );
    CREATE TABLE pedidos (
        pedido_id INTEGER PRIMARY KEY,
        cliente_id INTEGER NOT NULL REFERENCES clientes(cliente_id),
        fecha TEXT NOT NULL,
        estado TEXT NOT NULL DEFAULT 'pendiente',
        total REAL NOT NULL DEFAULT 0 CHECK (total >= 0)
    );
    CREATE TABLE detalle_pedido (
        pedido_id INTEGER NOT NULL REFERENCES pedidos(pedido_id) ON DELETE CASCADE,
        producto_id INTEGER NOT NULL REFERENCES productos(producto_id),
        cantidad INTEGER NOT NULL CHECK (cantidad > 0),
        precio_unit REAL NOT NULL CHECK (precio_unit >= 0),
        PRIMARY KEY (pedido_id, producto_id)
    );
    ''')
    print('✅ Esquema creado en línea')

print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn))

---
## 2. Cargar datos desde CSV

In [ ]:
def cargar_csv(tabla, archivo):
    path = DATA / archivo
    df = pd.read_csv(path)
    df.to_sql(tabla, conn, if_exists='append', index=False)
    print(f'{tabla}: {len(df)} filas')
    return df

# Orden respeta FKs
cargar_csv('categorias', 'categorias.csv')
cargar_csv('productos', 'productos.csv')
cargar_csv('clientes', 'clientes.csv')
cargar_csv('pedidos', 'pedidos.csv')
cargar_csv('detalle_pedido', 'detalle_pedido.csv')
conn.commit()

In [ ]:
# TODO 2.1: Verifica conteos y muestra 5 productos con su categoría (JOIN)
print(pd.read_sql('SELECT COUNT(*) AS n FROM clientes', conn))
# Tu consulta JOIN aquí


---
## 3. Consultas avanzadas

In [ ]:
# TODO 3.1: Top 10 clientes por monto total de pedidos entregados/enviados
q = '''
-- Escribe tu SQL aquí
SELECT 1 AS placeholder
'''
print(pd.read_sql(q, conn))

In [ ]:
# TODO 3.2: Ventas por categoría (nombre categoría, unidades, monto)
q = '''
SELECT 1 AS placeholder
'''
print(pd.read_sql(q, conn))

In [ ]:
# TODO 3.3: CTE — clientes con más de 2 pedidos y su ticket promedio
q = '''
WITH pedidos_cliente AS (
    SELECT cliente_id, COUNT(*) AS n, AVG(total) AS ticket
    FROM pedidos
    WHERE estado != 'cancelado'
    GROUP BY cliente_id
    HAVING COUNT(*) > 2
)
SELECT c.nombre, c.ciudad, p.n, ROUND(p.ticket, 2) AS ticket_promedio
FROM pedidos_cliente p
JOIN clientes c ON c.cliente_id = p.cliente_id
ORDER BY p.ticket DESC
'''
print(pd.read_sql(q, conn))

In [ ]:
# TODO 3.4: Window function — ranking de pedidos por cliente según total
# SQLite 3.25+ soporta window functions
q = '''
SELECT
    cliente_id,
    pedido_id,
    total,
    RANK() OVER (PARTITION BY cliente_id ORDER BY total DESC) AS rank_cliente
FROM pedidos
WHERE estado != 'cancelado'
ORDER BY cliente_id, rank_cliente
LIMIT 20
'''
print(pd.read_sql(q, conn))

---
## 4. Transacciones y CRUD

In [ ]:
# TODO 4.1: Implementa una función que registre un pedido de forma transaccional:
# - Verifica stock suficiente
# - Inserta en pedidos y detalle_pedido
# - Descuenta stock
# - Actualiza total del pedido
# - COMMIT o ROLLBACK

def crear_pedido(conn, cliente_id, items):
    """items: list of (producto_id, cantidad)"""
    try:
        conn.execute('BEGIN')
        # Tu lógica aquí
        conn.execute('COMMIT')
        return True
    except Exception as e:
        conn.execute('ROLLBACK')
        print('Error:', e)
        return False

print('Función definida. Prueba con crear_pedido(conn, 1, [(1, 1)])')

In [ ]:
# TODO 4.2: CRUD parametrizado — actualiza el email de un cliente de forma segura
cliente_id = 1
nuevo_email = 'nuevo@mail.com'
# conn.execute('UPDATE clientes SET email = ? WHERE cliente_id = ?', (nuevo_email, cliente_id))
# conn.commit()
print(pd.read_sql('SELECT * FROM clientes WHERE cliente_id = 1', conn))

---
## 5. Caso práctico: informe de ventas

Construye un informe SQL que responda:
1. ¿Cuáles son las 5 ciudades con mayor monto de ventas?
2. ¿Qué categoría tiene mayor rotación (unidades vendidas)?
3. ¿Cuál es el ticket promedio mensual?

Documenta cada consulta y exporta un resumen a CSV.

In [ ]:
# === CASO PRÁCTICO ===
# 1. Ciudades top

# 2. Categoría con más unidades

# 3. Ticket promedio por mes (substr(fecha,1,7) en SQLite)


---
## 6. Retos

1. **Índices:** usa `EXPLAIN QUERY PLAN` en una consulta lenta y justifica un índice.
2. **Integridad:** intenta insertar un pedido con `cliente_id` inexistente y observa el error de FK.
3. **MongoDB (opcional):** modela productos como documentos e implementa find + aggregate equivalentes.
4. **Seguridad:** reescribe una consulta vulnerable a inyección usando parámetros.

---

## ✅ Entrega

- Esquema correcto con FKs e índices
- Consultas avanzadas resueltas
- Transacción de pedido funcionando
- Informe del caso práctico documentado

*Prof. Fabián Soberanes Martín*

In [ ]:
conn.close()
print('Conexión cerrada.')